# CIR Interest Rate Model — FinClub Open Project 2026

**Goal:** Calibrate the Cox-Ingersoll-Ross (CIR) model on historical yield data and reconstruct the full yield curve (6M → 2Y) using only the 3-Month rate as input.  
**Target:** Out-of-sample R² > 0.85

**Workflow:**
1. Load and clean the data
2. Implement and calibrate the CIR model (MLE)
3. Predict the yield curve from the 3M rate
4. Extend with CIR++ (deterministic shift) to improve accuracy
5. Evaluate and analyse


## 1. Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from scipy.stats import ncx2
from sklearn.metrics import r2_score, mean_absolute_error

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print('Imports OK')

Imports OK


## 2. Data Loading and Preprocessing

The raw CSVs contain daily zero-coupon bond yields across 9 maturities.  
We need to:
- Parse dates and sort chronologically
- Fill missing values via time-interpolation
- Clip outliers using IQR (3× rule to preserve genuine regime shifts)
- Ensure all yields are positive (CIR requires r > 0)

In [ ]:
# Column names map to maturity in years
MATURITY_MAP = {
    'ZC025YR': 0.25,
    'ZC050YR': 0.50,
    'ZC075YR': 0.75,
    'ZC100YR': 1.00,
    'ZC200YR': 2.00,
    'ZC500YR': 5.00,
    'ZC1000YR': 10.00,
    'ZC2000YR': 20.00,
    'ZC3000YR': 30.00,
}

def preprocess(df, name):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date').sort_index()
    df = df.rename(columns={k: v for k, v in MATURITY_MAP.items() if k in df.columns})

    # Fill gaps: interpolate first, then forward/back fill edges
    n_missing = df.isnull().sum().sum()
    df = df.interpolate(method='time').ffill().bfill()
    print(f'{name}: filled {n_missing} missing values')

    # Winsorise outliers per column using 3×IQR
    for col in df.columns:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 3*iqr, q3 + 3*iqr
        n_clips = ((df[col] < lo) | (df[col] > hi)).sum()
        if n_clips:
            df[col] = df[col].clip(lo, hi)
            print(f'  {name} [{col}]: clipped {n_clips} outliers')

    df = df.clip(lower=1e-6)  # ensure positivity
    return df


train_raw  = pd.read_csv('train_data.csv')
test_raw   = pd.read_csv('test_data.csv')
test3m_raw = pd.read_csv('test_data_3M.csv')

train  = preprocess(train_raw,  'TRAIN')
test   = preprocess(test_raw,   'TEST')
test3m = preprocess(test3m_raw, 'TEST_3M')

print(f'\nTrain: {train.shape}, {train.index[0].date()} → {train.index[-1].date()}')
print(f'Test:  {test.shape},  {test.index[0].date()} → {test.index[-1].date()}')
print('Columns:', train.columns.tolist())

### Quick EDA

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
fig.suptitle('Training Yields by Maturity', fontsize=14, fontweight='bold')
labels = ['3M','6M','9M','1Y','2Y','5Y','10Y','20Y','30Y']
colors = plt.cm.tab10.colors

for ax, col, lbl, clr in zip(axes.flat, train.columns, labels, colors):
    ax.plot(train.index, train[col]*100, color=clr, linewidth=0.8)
    ax.set_title(lbl, fontsize=10)
    ax.set_ylabel('Yield (%)', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. CIR Model — Mathematics and Implementation

The CIR model describes the short rate $r_t$ by:

$$dr_t = \kappa(\theta - r_t)\,dt + \sigma\sqrt{r_t}\,dW_t$$

- $\kappa$ = speed of mean reversion
- $\theta$ = long-run mean rate
- $\sigma$ = volatility

**Feller condition:** $2\kappa\theta \geq \sigma^2$ keeps rates strictly positive.

**Zero-coupon bond price:**
$$P(t,T) = A(\tau)\,e^{-B(\tau)\,r_t}, \quad \tau = T-t$$

$$h = \sqrt{\kappa^2 + 2\sigma^2}$$
$$B(\tau) = \frac{2(e^{h\tau}-1)}{(h+\kappa)(e^{h\tau}-1)+2h}$$
$$A(\tau) = \left[\frac{2h\,e^{(h+\kappa)\tau/2}}{(h+\kappa)(e^{h\tau}-1)+2h}\right]^{\!2\kappa\theta/\sigma^2}$$

**Yield:**
$$y(\tau) = \frac{B(\tau)\,r_t - \ln A(\tau)}{\tau}$$

**Calibration:** We use MLE via the exact non-central chi-squared transition density of the CIR process. This is statistically optimal (asymptotically efficient) compared to OLS.

In [ ]:
class CIRModel:
    """
    Cox-Ingersoll-Ross short-rate model.
    Supports MLE calibration and closed-form yield curve construction.
    """

    def __init__(self, kappa=1.0, theta=0.02, sigma=0.05):
        self.kappa = kappa
        self.theta = theta
        self.sigma = sigma

    # ── Bond pricing functions ──────────────────────────────────────────────

    def _h(self):
        return np.sqrt(self.kappa**2 + 2*self.sigma**2)

    def B(self, tau):
        h = self._h()
        e = np.exp(h * tau)
        return 2*(e-1) / ((h+self.kappa)*(e-1) + 2*h)

    def A(self, tau):
        h = self._h()
        e = np.exp(h * tau)
        numer = 2*h * np.exp((h+self.kappa)*tau/2)
        denom = (h+self.kappa)*(e-1) + 2*h
        exp = 2*self.kappa*self.theta / self.sigma**2
        return (numer/denom)**exp

    def yield_curve(self, tau, r):
        """Continuously compounded yield for maturity tau, given short rate r."""
        tau = np.atleast_1d(tau).astype(float)
        return (self.B(tau)*r - np.log(self.A(tau))) / tau

    # ── Calibration ─────────────────────────────────────────────────────────

    @staticmethod
    def _neg_loglik(params, r, dt=1/252):
        """Negative log-likelihood using exact CIR transition (non-central chi-squared)."""
        kappa, theta, sigma = params
        if kappa <= 0 or theta <= 0 or sigma <= 0:
            return 1e10

        r_t  = r[:-1]
        r_t1 = r[1:]

        c  = 2*kappa / (sigma**2 * (1 - np.exp(-kappa*dt)))
        df = 4*kappa*theta / sigma**2
        nc = 2*c * np.exp(-kappa*dt) * r_t
        u  = 2*c * r_t1

        log_p = ncx2.logpdf(u, df=df, nc=nc) + np.log(2*c)
        valid = np.isfinite(log_p)
        return -log_p[valid].sum() if valid.sum() > 10 else 1e10

    def _ols_init(self, r, dt=1/252):
        """Fast OLS warm-start for MLE."""
        r_t, r_t1 = r[:-1], r[1:]
        dr = r_t1 - r_t
        X = np.column_stack([np.ones_like(r_t), r_t])
        beta, *_ = np.linalg.lstsq(X, dr, rcond=None)
        a, b = beta
        kappa = max(-b/dt, 1e-4)
        theta = max(a/(kappa*dt), 1e-6)
        res = dr - a - b*r_t
        sigma = np.sqrt(max(np.var(res) / (np.mean(r_t)*dt), 1e-8))
        return [kappa, theta, sigma]

    def calibrate(self, r, dt=1/252):
        """Calibrate (κ, θ, σ) via MLE with global search (differential evolution)."""
        r = np.asarray(r, dtype=float)
        r = r[r > 1e-8]

        x0 = self._ols_init(r, dt)
        bounds = [(0.01, 20.0), (0.001, 0.20), (0.001, 0.50)]

        # Global search to avoid local minima
        de = differential_evolution(
            self._neg_loglik, bounds=bounds, args=(r, dt),
            seed=42, maxiter=300, tol=1e-9, polish=True,
            mutation=(0.5, 1.5), recombination=0.9, popsize=15
        )
        # Local refinement
        res = minimize(
            self._neg_loglik, de.x, args=(r, dt),
            method='Nelder-Mead',
            options={'maxiter': 10000, 'xatol': 1e-12, 'fatol': 1e-12}
        )
        self.kappa, self.theta, self.sigma = res.x
        return self

    def summary(self):
        feller = 2*self.kappa*self.theta / self.sigma**2
        print(f'  κ = {self.kappa:.5f}  (half-life = {np.log(2)/self.kappa:.2f} yrs)')
        print(f'  θ = {self.theta:.5f}  ({self.theta*100:.3f}%)')
        print(f'  σ = {self.sigma:.5f}')
        print(f'  Feller 2κθ/σ² = {feller:.4f}  ({"OK" if feller >= 1 else "VIOLATED"})')


print('CIRModel defined')

In [ ]:
# Calibrate on the 3M short-rate series from training data
r_train = train[0.25].values

print('Calibrating CIR via MLE (differential evolution + Nelder-Mead)...')
cir = CIRModel()
cir.calibrate(r_train)

print('\nCalibrated parameters:')
cir.summary()

In [ ]:
# Plot a few simulated CIR paths vs the actual 3M rate to check calibration quality
np.random.seed(42)
T, dt = len(r_train), 1/252
n_paths = 8

r_sim = np.zeros((T, n_paths))
r_sim[0] = r_train[0]

for t in range(1, T):
    rp = r_sim[t-1]
    dW = np.random.normal(0, np.sqrt(dt), n_paths)
    r_sim[t] = np.maximum(
        rp + cir.kappa*(cir.theta - rp)*dt + cir.sigma*np.sqrt(np.maximum(rp,0))*dW, 0
    )

fig, ax = plt.subplots(figsize=(14, 5))
for i in range(n_paths):
    ax.plot(train.index, r_sim[:,i]*100, alpha=0.2, linewidth=0.7, color='steelblue')
ax.plot(train.index, r_train*100, color='orange', linewidth=1.5, label='Actual 3M')
ax.plot(train.index, r_sim.mean(axis=1)*100, color='green', linewidth=1.2,
        linestyle='--', label='Sim mean')
ax.axhline(cir.theta*100, color='violet', linestyle=':', linewidth=1.5,
           label=f'θ = {cir.theta*100:.2f}%')
ax.set(xlabel='Date', ylabel='Rate (%)', title='CIR Simulated Paths vs Actual 3M Rate')
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 4. Yield Curve Prediction Challenge

For each test day, we are given **only the 3M yield** as the proxy for $r_t$.  
We reconstruct the yield curve at maturities 6M, 9M, 1Y, 2Y using the closed-form CIR formula.

The test period is **completely unseen** — strict out-of-sample evaluation.

In [ ]:
TARGET_MATURITIES = [0.50, 0.75, 1.00, 2.00]
TARGET_LABELS     = ['6M', '9M', '1Y', '2Y']
taus = np.array(TARGET_MATURITIES)

def predict_yields(model, r_3m_series):
    """Build predicted yield DataFrame from 3M rate series."""
    rows = {}
    for date, r in r_3m_series.items():
        rows[date] = model.yield_curve(taus, r)
    pred = pd.DataFrame(rows, index=TARGET_MATURITIES).T
    pred.index.name = 'Date'
    pred.columns = TARGET_MATURITIES
    return pred


def evaluate(pred, actual, model_name='Model'):
    """Print R², MAE per maturity and return overall R²."""
    common = pred.index.intersection(actual.index)
    all_p, all_a = [], []

    print(f'\n{"─"*55}')
    print(f'  {model_name}  —  Out-of-Sample Performance')
    print(f'{"─"*55}')
    print(f'  {"Tenor":<8} {"R²":>8} {"MAE (bps)":>12}')
    print(f'{"─"*55}')

    for tau, lbl in zip(TARGET_MATURITIES, TARGET_LABELS):
        y_p = pred.loc[common, tau].values
        y_a = actual.loc[common, tau].values
        r2  = r2_score(y_a, y_p)
        mae = mean_absolute_error(y_a, y_p) * 10000
        print(f'  {lbl:<8} {r2:>8.4f} {mae:>10.2f}')
        all_p.extend(y_p)
        all_a.extend(y_a)

    overall = r2_score(all_a, all_p)
    overall_mae = mean_absolute_error(all_a, all_p) * 10000
    print(f'{"─"*55}')
    print(f'  {"OVERALL":<8} {overall:>8.4f} {overall_mae:>10.2f}')
    target_met = '✅ PASS' if overall > 0.85 else '❌ FAIL'
    print(f'  Target R² > 0.85: {target_met}')
    print(f'{"─"*55}')

    return overall


r_test = test3m[0.25]
pred_base = predict_yields(cir, r_test)

r2_base = evaluate(pred_base, test, 'Base CIR (MLE)')

In [ ]:
# Plot predicted vs actual for each maturity
common = pred_base.index.intersection(test.index)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Base CIR — Predicted vs Actual Yields (Test Period)', fontsize=13, fontweight='bold')

for ax, tau, lbl in zip(axes.flat, TARGET_MATURITIES, TARGET_LABELS):
    y_true = test.loc[common, tau]
    y_pred = pred_base.loc[common, tau]
    r2 = r2_score(y_true, y_pred)

    ax.plot(common, y_true*100, color='orange', linewidth=1.4, label='Actual')
    ax.plot(common, y_pred*100, color='steelblue', linewidth=1.1,
            linestyle='--', label='Predicted')
    ax.fill_between(common, y_true*100, y_pred*100, alpha=0.2, color='red')
    ax.set_title(f'{lbl}  R²={r2:.4f}', fontsize=11)
    ax.set_ylabel('Yield (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.show()

## 5. Extension: CIR++ (Deterministic Shift)

The base CIR cannot perfectly fit an arbitrary yield curve — the long-run mean $\theta$ pulls all maturities toward a single attractor, causing systematic bias.

**CIR++ idea (Brigo & Mercurio, 2001):**  
Add a maturity-specific constant shift $\varphi(\tau)$ bootstrapped from training residuals:

$$y^{++}(\tau, r_t) = y^{\text{CIR}}(\tau, r_t) + \varphi(\tau)$$

where $\varphi(\tau) = \text{median}\left[y^{\text{market}}(\tau) - y^{\text{CIR}}(\tau, r_t)\right]$ over training.

**Why this works without overfitting:** $\varphi(\tau)$ is a single unconditional constant per maturity — it corrects the structural bias, not the day-to-day variance.

In [ ]:
class CIRPlusPlus:
    """
    CIR++ extension: adds a maturity-specific deterministic shift to the base CIR yield.
    The shift is the median training residual — robust and non-overfitting.
    """

    def __init__(self, base_model):
        self.base = base_model
        self.phi  = {}

    def fit(self, train_df, target_maturities):
        """Estimate φ(τ) for each target maturity from training data."""
        r_tr = train_df[0.25].values
        print('CIR++ shifts φ(τ) [bps]:')
        for tau in target_maturities:
            if tau not in train_df.columns:
                continue
            y_cir = np.array([self.base.yield_curve(np.array([tau]), r)[0] for r in r_tr])
            y_mkt = train_df[tau].values
            self.phi[tau] = np.median(y_mkt - y_cir)
            print(f'  τ={tau}: {self.phi[tau]*10000:+.1f} bps')
        return self

    def predict(self, r_3m_series, target_maturities):
        """Predict yields with CIR++ correction."""
        tau_arr = np.array(target_maturities)
        rows = {}
        for date, r in r_3m_series.items():
            y = self.base.yield_curve(tau_arr, r)
            y += np.array([self.phi.get(t, 0.0) for t in target_maturities])
            rows[date] = y
        pred = pd.DataFrame(rows, index=target_maturities).T
        pred.index.name = 'Date'
        pred.columns = target_maturities
        return pred


cirpp = CIRPlusPlus(cir)
cirpp.fit(train, TARGET_MATURITIES)

pred_pp = cirpp.predict(r_test, TARGET_MATURITIES)
r2_pp = evaluate(pred_pp, test, 'CIR++ (MLE + Shift)')

In [ ]:
# Head-to-head comparison: Base CIR vs CIR++
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Base CIR vs CIR++ — Test Period', fontsize=13, fontweight='bold')

for ax, tau, lbl in zip(axes.flat, TARGET_MATURITIES, TARGET_LABELS):
    y_true = test.loc[common, tau]
    y_base = pred_base.loc[common, tau]
    y_pp   = pred_pp.loc[common, tau]

    r2_b = r2_score(y_true, y_base)
    r2_p = r2_score(y_true, y_pp)

    ax.plot(common, y_true*100, color='orange', linewidth=1.6, label='Actual')
    ax.plot(common, y_base*100, color='steelblue', linewidth=1.0,
            linestyle='--', label=f'Base CIR R²={r2_b:.3f}')
    ax.plot(common, y_pp*100, color='green', linewidth=1.0,
            linestyle='-.', label=f'CIR++   R²={r2_p:.3f}')
    ax.set_title(lbl, fontsize=11)
    ax.set_ylabel('Yield (%)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.show()

## 6. Critical Analysis

### Calibration Method: Why MLE?

OLS treats the CIR process like a simple linear regression on $\Delta r_t$, matching only the conditional mean. The CIR transition density is exactly known — a scaled non-central chi-squared — so MLE extracts information from the **full distribution**, not just the mean. This makes MLE asymptotically efficient (minimum variance among consistent estimators).

### Sensitivity to Parameters

- **$\kappa$ (speed):** Controls the half-life of rate shocks ($\ln 2 / \kappa$). High $\kappa$ means rates quickly snap back to $\theta$. Low $\kappa$ implies persistent deviations — consistent with the prolonged low-rate era 2016–2022.
- **$\theta$ (long-run mean):** The asymptote of the yield curve at long maturities. Calibrated from the full training period, so it averages over multiple rate regimes.
- **$\sigma$ (volatility):** Affects the curvature of the term structure. Higher $\sigma$ flattens the curve.

### Feller Condition in Practice

During the near-zero rate environment (2016–2021), the short rate came close to the boundary. The Feller ratio may dip below 1 in rolling windows. In practice, we handle this by clipping simulated rates at a small positive floor ($10^{-6}$) and noting that MLE naturally avoids extreme parameter combinations via the likelihood penalisation.

### Hardest Maturities

The **2Y tenor** is typically hardest to fit. It sits at the boundary between monetary policy expectations (short end) and structural term premia (long end), making it the most sensitive to regime changes.

### CIR++ vs Alternatives

| Extension | Pros | Cons |
|-----------|------|------|
| **CIR++** | Tractable, exact fit to training term structure, 1 parameter per maturity | Shift is constant — can't adapt to changing regime bias |
| Two-factor CIR | Captures level + slope independently | Second factor unobservable, needs Kalman filter |
| Jump-diffusion | Models policy shocks explicitly | Needs high-frequency data to identify jump distribution |

We chose CIR++ because it is directly identifiable from a single observable (3M rate), parsimonious, and addresses the main failure mode of the base model.

### Remaining Limitations

1. **Single factor:** One Brownian motion cannot independently control level, slope, and curvature of the curve.
2. **Constant parameters:** $\kappa$, $\theta$, $\sigma$ are time-invariant; real rates undergo regime changes.
3. **Physical vs risk-neutral measure:** Calibration is under $\mathbb{P}$; derivatives pricing requires the $\mathbb{Q}$ measure with a market price of risk.
4. **3M proxy:** The 3-Month yield is not the instantaneous short rate; it introduces a small downward bias at short maturities.

In [ ]:
# Sensitivity: how yield curve changes as we perturb each parameter
tau_grid = np.linspace(0.25, 2.0, 100)
r_ref = cir.theta  # evaluate at the long-run mean
mults = [0.5, 0.75, 1.0, 1.25, 1.5]
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(mults)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Yield Curve Sensitivity to CIR Parameters', fontsize=12, fontweight='bold')

for ax, param in zip(axes, ['kappa', 'theta', 'sigma']):
    for mult, clr in zip(mults, colors):
        kw = {'kappa': cir.kappa, 'theta': cir.theta, 'sigma': cir.sigma}
        kw[param] = getattr(cir, param) * mult
        m = CIRModel(**kw)
        ax.plot(tau_grid, m.yield_curve(tau_grid, r_ref)*100,
                color=clr, linewidth=1.5, label=f'{mult}×')
    ax.set_title(f'Varying {param}', fontsize=10)
    ax.set_xlabel('Maturity (years)')
    ax.set_ylabel('Yield (%)')
    ax.legend(fontsize=8, title='mult')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Rolling Feller condition to see when it might be violated
window = 252
feller_vals, feller_dates = [], []

for start in range(0, len(r_train) - window, 21):
    r_w = r_train[start:start+window]
    try:
        m = CIRModel()
        x0 = m._ols_init(r_w)
        k, th, s = x0
        feller_vals.append(2*k*th / s**2)
        feller_dates.append(train.index[start + window//2])
    except:
        pass

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(feller_dates, feller_vals, color='steelblue', linewidth=1.2)
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='Feller boundary')
ax.fill_between(feller_dates, 0, 1, alpha=0.1, color='red', label='Feller violated')
ax.set(xlabel='Date', ylabel='2κθ/σ²', title='Rolling Feller Ratio (1Y window, OLS)')
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

pct = np.mean(np.array(feller_vals) < 1) * 100
print(f'Feller violated in {pct:.1f}% of rolling windows (mostly during near-zero rate era)')

## 7. Final Results Summary

In [ ]:
print('=' * 60)
print('  FINAL RESULTS')
print('=' * 60)
print('\nCalibrated CIR Parameters (MLE):')
cir.summary()

print('\nCIR++ Shifts:')
for tau, lbl in zip(TARGET_MATURITIES, TARGET_LABELS):
    print(f'  {lbl}: {cirpp.phi.get(tau, 0)*10000:+.1f} bps')

print('\n' + '─' * 60)
print(f'  {"Model":<20} {"Overall R²":>12}  Status')
print('─' * 60)
for name, r2 in [("Base CIR (MLE)", r2_base), ("CIR++ (MLE+Shift)", r2_pp)]:
    status = '✅ PASS' if r2 > 0.85 else '❌ FAIL'
    print(f'  {name:<20} {r2:>12.4f}  {status}')
print('─' * 60)

best_r2 = max(r2_base, r2_pp)
best = 'CIR++' if r2_pp >= r2_base else 'Base CIR'
print(f'\n🏆 Best model: {best} — R² = {best_r2:.4f}')
print(f'   R² > 0.85: {"✅ ACHIEVED" if best_r2 > 0.85 else "❌ NOT MET"}')
print('=' * 60)